In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from data_processing.processing.figure_of_merit import fit_fom, FOM, gaussian, n_sigma_classifier, bimodal
from scipy.

In [ ]:
data_file_path = Path("C:/Users/oliver.horner/Desktop/FOM Analysis 2024-05/vaporWave.parquet")

In [ ]:
psd_report = pd.read_parquet(data_file_path)
psd_report = psd_report.rename(columns={"vaporWave1": "CALIB_ENERGY", "vaporWave2": "PSD"})
psd_report = psd_report.dropna()
psd_report = psd_report[psd_report["PSD"].between(0, 0.5)]
psd_report

In [ ]:
# do histogram
# get energy bin bounds
x, y = psd_report["CALIB_ENERGY"], psd_report["PSD"]
e_bin_width = 15
e_bins = np.linspace(0, x.max(), int(x.max()/e_bin_width))
psd_bin_count = 100
psd_bins = np.linspace(0, 0.5, psd_bin_count+1)

Z, xe, ye = np.histogram2d(x, y, bins=[e_bins, psd_bins])

In [ ]:
# vaporwave island
# Graph 2D histogram as 3D contour plot (like a geographic height map)
# Allows view of the entire dataset's "shape"
contour_res = 60  # number of contour lines to use
angle_elev = 55  # Elevation angle (-90=underneath, 0=side, 90=above)
angle_rot = -60  # Rotation angle (0=down X (energy) axis, + rotates clockwise)
cmap = plt.colormaps["nipy_spectral"]

fig = plt.figure(figsize=(12, 12))
ax = plt.axes(projection='3d')

x, y = np.meshgrid(xe[:-1], ye[:-1])

ax.view_init(angle_elev, angle_rot)
ax.contour3D(x, y, Z.T, contour_res, cmap=cmap)

# ax.plot_surface(x, y, Z.T, cmap=cmap, antialiased=True)
# ax.zaxis.set_major_locator(ticker.LogLocator())

# ax.contour([0.35, 0.35], [0, 0], [100, 100])
ax.set_ylim(0, 0.5)
ax.set_title("PSD Histogram", fontsize=16)
ax.set_ylabel("PSD", fontsize=16)
ax.set_xlabel("Energy (keVee)", fontsize=16)
ax.set_zlabel("Counts", fontsize=16)

In [ ]:
# FOM function definition

BimodalParams = tuple[float, float, float, float, float, float]
GaussianParams = tuple[float, float, float]
BimodalBounds = tuple[BimodalParams, BimodalParams]

def split_params(
    params: BimodalParams
) -> tuple[GaussianParams, GaussianParams]:
    """Separates bimodal function parameters into 2 sets, one per component gaussian
    
    Parameters
    ----------
    params: BimodalParams
        parameters of a bimodal function
        
    Returns
    -------
    lower_gaussian_params: GaussianParams
        parameters of the lower gaussian (i.e. lower mu value)
    upper_gaussian_params: GaussianParams
        parameters of the upper gaussian (i.e. higher mu value)
    """
    params = abs(params)
    return params[0:3], params[3:]


def get_bimodal_fit(
    bins: np.ndarray, 
    histogram_slice: np.ndarray, 
    bounds: BimodalBounds | None = None,
    initial_guess: BimodalParams | None = None
) -> tuple[GaussianParams, GaussianParams, np.ndarray]:
    """Fits a histogram slice to a bimodal distribution
    
    Parameters
    ----------
    bins: ndarray
        lower bounds of each PSD bin in the histogram
    histogram_slice: ndarray
        slice of the 2D PSD/Energy histogram taken for a specific energy (i.e. PSD vs Counts)
    bounds: BimodalBounds
        lower and upper bounds of fit parameters for this slice
        
    Returns
    -------
    gamma_params: GaussianParams
        parameters of the gaussian fit for gamma rays
    neutron_params: GaussianParams
        parameters of the gaussian fit for neutrons
    cov: ndarray
        estimated covariance of all bimodial parameters
    """
    if bounds is None:
        bounds = (-np.inf, np.inf)
    params, cov = curve_fit(
        bimodal,
        bins,
        histogram_slice,
        bounds=bounds,
        p0=initial_guess
    )
    gamma_params, neutron_params = split_params(params)
    
    return gamma_params, neutron_params, cov


def scan_histogram_slices(
    bins: np.ndarray, 
    histogram: np.ndarray, 
    default_bounds: BimodalBounds,
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None = None, 
    start_idx: int = 0, 
    end_idx: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Determines bimodal fit and FOM for every energy slice in 2D PSD/Energy histogram
    
    Parameters
    ----------
    bins: ndarray
        lower bounds of each PSD bin in the histogram
    histogram: ndarray
        2D PSD/Energy histogram
    default_bounds: BimodalBounds
        default lower and upper bounds of fit parameters
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None, default None
        allows custom bounds for slice ranges. 
        Each list entry must have a tuple of start and stop indexes, and corresponding fit bounds.
        Bounds are used when the slice index falls within the start/stop range (start inclusive, stop exclusive).
        If index ranges overlap, the last matching range is used.
        If bounds is None, only default_bounds are used.
    start_idx: int, default 0
        starting index (inclusive) of slice range to fit to bimodal
    end_idx: int | None, default None
        ending index (exclusive) of slice range to fit to bimodal
        
    Returns
    -------
    fit_dataframe: DataFrame
        DataFrame of fit parameters including FOM (as columns) for each slice (as rows)
    error_dataframe: DataFrame
        DataFrame of (1 standard deviation) errors in fit parameters (as columns) for each slice (as rows)
    """
    end_idx = len(histogram) if end_idx is None else min(len(histogram), end_idx)
        
    slice_params = []
    slice_err = []
    
    for i in range(start_idx, end_idx):
        fit_bounds = default_bounds
        
        if bounds is not None :
            for i_range, bound in bounds:
                if i in range(*i_range):
                    fit_bounds = bound
        
#         print(f"default = {default_bounds}\nbounds={bounds}")
        gamma_params, neutron_params, cov = get_bimodal_fit(bins, histogram[:,i], fit_bounds)
        
        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])
        
        perr = np.sqrt(np.diag(cov))
    
        slice_params.append((i, *gamma_params, *neutron_params, fom))
        slice_err.append((i, *perr))
    
    columns = ['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2']
    df = pd.DataFrame(slice_params, columns=columns + ['fom'])
    err_df = pd.DataFrame(slice_err, columns=columns)
    
    return df, err_df


def scan_histogram_slices_new(
    bins: np.ndarray,
    histogram: np.ndarray,
    default_bounds: BimodalBounds,
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None = None,
    start_idx: int = 0,
    end_idx: int | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Determines bimodal fit and FOM for every energy slice in 2D PSD/Energy histogram

    Parameters
    ----------
    bins: ndarray
        lower bounds of each PSD bin in the histogram
    histogram: ndarray
        2D PSD/Energy histogram
    default_bounds: BimodalBounds
        default lower and upper bounds of fit parameters
    bounds: list[tuple[tuple[int, int], BimodalBounds]] | None, default None
        allows custom bounds for slice ranges. 
        Each list entry must have a tuple of start and stop indexes, and corresponding fit bounds.
        Bounds are used when the slice index falls within the start/stop range (start inclusive, stop exclusive).
        If index ranges overlap, the last matching range is used.
        If bounds is None, only default_bounds are used.
    start_idx: int, default 0
        starting index (inclusive) of slice range to fit to bimodal
    end_idx: int | None, default None
        ending index (exclusive) of slice range to fit to bimodal

    Returns
    -------
    fit_dataframe: DataFrame
        DataFrame of fit parameters including FOM (as columns) for each slice (as rows)
    error_dataframe: DataFrame
        DataFrame of (1 standard deviation) errors in fit parameters (as columns) for each slice (as rows)
    """
    end_idx = len(histogram) if end_idx is None else min(len(histogram), end_idx)

    slice_params = []
    slice_err = []

    for i in range(start_idx, end_idx):
        fit_bounds = default_bounds

        if bounds is not None:
            for i_range, bound in bounds:
                if i in range(*i_range):
                    fit_bounds = bound

        slice_data = histogram[:, i]
        amp_guess = np.max(slice_data)
        # mean_guess = np.mean(slice_data)
        # sigma_guess = np.std(slice_data)
        initial_guess = (
            0.15, 0.05, amp_guess,
            0.35, 0.025, amp_guess
        )

#         print(f"default = {default_bounds}\nbounds={bounds}")
        gamma_params, neutron_params, cov = get_bimodal_fit(
            bins,
            slice_data,
            initial_guess=initial_guess,
            bounds=fit_bounds
        )

        fom = FOM(*gamma_params[:-1], *neutron_params[:-1])

        perr = np.sqrt(np.diag(cov))

        slice_params.append((i, *gamma_params, *neutron_params, fom))
        slice_err.append((i, *perr))

    columns = ['i', 'mu1', 'sigma1', 'a1', 'mu2', 'sigma2', 'a2']
    df = pd.DataFrame(slice_params, columns=columns + ['fom'])
    err_df = pd.DataFrame(slice_err, columns=columns)

    return df, err_df

In [ ]:
psd_bin_lbs = ye[:-1]
psd_bin_ubs = ye[1:]
psd_bin_centers = [(lb+ub)/2 for lb, ub in zip(psd_bin_lbs, psd_bin_ubs)]

# Default
default_bounds = (
    (0.1, 0.01, 1, 
     0.25, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.38, 0.04, Z.max())
)

bounds_a = (
    (0.1, 0.01, 1, 
     0.33, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.04, Z.max())
)

bounds_b = (
    (0.1, 0.01, 1, 
     0.34, 0.01, 0),
    (0.2, 0.1, Z.max(),
     0.36, 0.03, 4000)
)


# Ranged Example
bounds = [
    ((0,60), bounds_a),
]

start_scan_idx = 0
end_scan_idx = 420

end_scan_idx = min(end_scan_idx, len(Z))

# df_fom = find_threshold_fom_slice(psd_bin_lbs, Z.T, bounds, 0, end_scan_idx)

fit_df, fit_df_err = scan_histogram_slices_new(
    psd_bin_centers, 
    Z.T,
    bounds=bounds,
    default_bounds=default_bounds, 
    start_idx = start_scan_idx, 
    end_idx = end_scan_idx
)
# fit_df, fit_df_err = scan_histogram_slices_no_bounds(
#     psd_bin_centers, 
#     Z.T, 
#     start_idx=start_scan_idx, 
#     end_idx=end_scan_idx
# )

fit_df.head()

In [ ]:
fit_df['e_lower_bound'] = xe[fit_df['i']]
fit_df['e_upper_bound'] = xe[fit_df['i']+1]
fit_df

In [ ]:
slice_idx = 19
fom_slice = Z.T[:,slice_idx]
fom_slice_energy = (xe[slice_idx]+xe[slice_idx+1])/2
fom_curve_params = tuple(fit_df.iloc[slice_idx,1:7])
fom_row = fit_df.iloc[slice_idx,:]
fom = fom_row["fom"]

fom_row

In [ ]:
fom_slice[38]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.bar(psd_bin_lbs, fom_slice, width=0.005, color="black")
ax.plot(psd_bin_lbs, bimodal(psd_bin_lbs, *fom_curve_params), "r--", lw=4)
ax.grid()
ax.set_title(f"E={fom_slice_energy:.3f} keVee, FOM={fom:.3f}")

In [ ]:
psd_report

In [ ]:
data_file_folder = data_file_path.parent
psd_report_path = data_file_folder / "psd_data.csv"

psd_report.to_csv(psd_report_path, index=False)

In [ ]:
fom_analysis_path = data_file_folder / "fom_analysis.csv"
fit_df.to_csv(fom_analysis_path, index=False)